In [ ]:

import re
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform as rio_transform

# CONFIGURAÇÃO

SAR_ROOT = Path(r"resultados_petrobras_07.08-20260810T133647Z-1-001\resultados_petrobras_07.08\2")
LIDAR_XLSX = Path(r"LIDAR_PRA-1\lidar.xlsx")
OUTPUT_XLSX = Path(r"comparacao_XMOD_Lidar_3rian.xlsx")

LIDAR_LAT = -22.171268   # 22,171268° S
LIDAR_LON = -40.121612   # 40,121612° W

XMOD_MODELS = [2, 3]
SMOOTH_KS = [0, 9, 21]

LIDAR_TIME_COL = "Timestamp (end of interval)"
LIDAR_SPEED_COL = "40m Wind Speed (m/s)"
LIDAR_DIR_COL = "40m Wind Direction (°)"

MAX_TIME_DIFF = timedelta(minutes=30)

FOLDER_PATTERN = re.compile(r"^(\d{8})_(\d{6})$")

# FUNÇÕES UTILITÁRIAS

def parse_folder_timestamp(folder_name: str) -> datetime | None:
    """Extrai o timestamp (assumido UTC) a partir do nome da pasta."""
    m = FOLDER_PATTERN.match(folder_name)
    if not m:
        return None
    date_str, time_str = m.groups()
    return datetime.strptime(date_str + time_str, "%Y%m%d%H%M%S")


def read_value_at_point(tif_path: Path, lat: float, lon: float) -> float:
    """
    Lê o valor do pixel mais próximo às coordenadas (lat, lon) em um GeoTIFF,
    convertendo automaticamente de WGS84 para o CRS do raster.
    Retorna np.nan se o arquivo não existir, o ponto estiver fora da imagem,
    ou o valor for nodata.
    """
    if not tif_path.exists():
        print(f"  [aviso] arquivo não encontrado: {tif_path.name}")
        return np.nan

    with rasterio.open(tif_path) as src:
        xs, ys = rio_transform("EPSG:4326", src.crs, [lon], [lat])
        x, y = xs[0], ys[0]

        row, col = src.index(x, y)
        if not (0 <= row < src.height and 0 <= col < src.width):
            print(f"  [aviso] ponto fora da extensão da imagem: {tif_path.name}")
            return np.nan

        value = src.read(1, window=((row, row + 1), (col, col + 1)))[0, 0]

        nodata = src.nodata
        if nodata is not None and value == nodata:
            return np.nan
        if np.isnan(value):
            return np.nan

        return float(value)


def compute_direction_from_uv(u: float, v: float) -> float:
    """
    Direção meteorológica do vento (de onde o vento vem), em graus,
    0° = Norte, sentido horário — a partir das componentes u10 (leste)
    e v10 (norte). Ajuste a fórmula se a convenção dos seus dados for outra
    (ex.: convenção oceanográfica "para onde vai").
    """
    if np.isnan(u) or np.isnan(v):
        return np.nan
    return float(np.degrees(np.arctan2(-u, -v)) % 360)


def find_day_folders(root: Path):
    """Retorna lista de (timestamp, pasta) para todas as subpastas válidas."""
    entries = []
    for entry in sorted(root.iterdir()):
        if entry.is_dir():
            ts = parse_folder_timestamp(entry.name)
            if ts is not None:
                entries.append((ts, entry))
    return entries


def match_lidar_row(sar_time: datetime, lidar_df: pd.DataFrame):
    """Encontra a linha do Lidar com timestamp mais próximo, dentro da tolerância."""
    diffs = (lidar_df[LIDAR_TIME_COL] - sar_time).abs()
    idx_min = diffs.idxmin()
    if diffs.loc[idx_min] > MAX_TIME_DIFF:
        return None
    return lidar_df.loc[idx_min]

# PROCESSAMENTO PRINCIPAL

def main():
    lidar_df = pd.read_excel(LIDAR_XLSX)
    lidar_df[LIDAR_TIME_COL] = pd.to_datetime(lidar_df[LIDAR_TIME_COL])

    day_folders = find_day_folders(SAR_ROOT)
    if not day_folders:
        raise RuntimeError(f"Nenhuma pasta de dia encontrada em {SAR_ROOT}")

    intensity_rows = []
    direction_rows = []

    for sar_time, folder in day_folders:
        print(f"Processando {folder.name} ...")
        intensity_row = {"imagem": folder.name, "timestamp": sar_time}
        direction_row = {"imagem": folder.name, "timestamp": sar_time}

        for n in XMOD_MODELS:
            for k in SMOOTH_KS:
                col = f"XMOD{n}_smooth{k}"

                # --- Intensidade ---
                speed_path = folder / f"iceye_ecmwf_wind_XMOD{n}_smooth{k}.tif"
                intensity_row[col] = read_value_at_point(speed_path, LIDAR_LAT, LIDAR_LON)

                # --- Direção, calculada a partir de u10 e v10 do próprio XMOD/K ---
                u_path = folder / f"iceye_ecmwf_wind_XMOD{n}_u10_smooth{k}.tif"
                v_path = folder / f"iceye_ecmwf_wind_XMOD{n}_v10_smooth{k}.tif"
                u_val = read_value_at_point(u_path, LIDAR_LAT, LIDAR_LON)
                v_val = read_value_at_point(v_path, LIDAR_LAT, LIDAR_LON)
                direction_row[col] = compute_direction_from_uv(u_val, v_val)

        # --- Lidar ---
        lidar_match = match_lidar_row(sar_time, lidar_df)
        if lidar_match is not None:
            intensity_row["Lidar"] = lidar_match[LIDAR_SPEED_COL]
            direction_row["Lidar"] = lidar_match[LIDAR_DIR_COL]
        else:
            print(f"  [aviso] nenhum registro do Lidar dentro da tolerância de tempo")
            intensity_row["Lidar"] = np.nan
            direction_row["Lidar"] = np.nan

        intensity_rows.append(intensity_row)
        direction_rows.append(direction_row)

    intensity_df = pd.DataFrame(intensity_rows).set_index("imagem")
    direction_df = pd.DataFrame(direction_rows).set_index("imagem")

    OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        intensity_df.to_excel(writer, sheet_name="Intensidade")
        direction_df.to_excel(writer, sheet_name="Direcao")

    print(f"\nTabelas salvas em: {OUTPUT_XLSX}")
    print("\n--- Intensidade ---")
    print(intensity_df)
    print("\n--- Direção ---")
    print(direction_df)


if __name__ == "__main__":
    main()

FileNotFoundError: [WinError 3] O sistema não pode encontrar o caminho especificado: 'XMOD\\resultados petrobras 07.08-20260810T133647Z-1-001\\resultados petrobras 07.08\\2º'

In [2]:
import pandas as pd
df = pd.read_excel(r"LIDAR_PRA-1\lidar.xlsx")
print(df.columns.tolist())
print(df.head())

['Timestamp (end of interval)', 'Int Temp (°C)', 'Ext Temp (°C)', 'Pressure (hPa)', 'Rel Humidity (%)', 'Wiper count', 'Vbatt (V)', '40m Wind Speed (m/s)', '40m Wind Speed Dispersion (m/s)', '40m Wind Speed min (m/s)', '40m Wind Speed max (m/s)', '40m Wind Direction (°)', '40m Z-wind (m/s)', '40m Z-wind Dispersion (m/s)', '40m CNR (dB)', '40m CNR min (dB)', '40m Dopp Spect Broad (m/s)', '40m Data Availability (%)', 'Coluna1', '50m Wind Speed (m/s)', '50m Wind Speed Dispersion (m/s)', '50m Wind Speed min (m/s)', '50m Wind Speed max (m/s)', '50m Wind Direction (°)', '50m Z-wind (m/s)', '50m Z-wind Dispersion (m/s)', '50m CNR (dB)', '50m CNR min (dB)', '50m Dopp Spect Broad (m/s)', '50m Data Availability (%)', 'Coluna2', '60m Wind Speed (m/s)', '60m Wind Speed Dispersion (m/s)', '60m Wind Speed min (m/s)', '60m Wind Speed max (m/s)', '60m Wind Direction (°)', '60m Z-wind (m/s)', '60m Z-wind Dispersion (m/s)', '60m CNR (dB)', '60m CNR min (dB)', '60m Dopp Spect Broad (m/s)', '60m Data Avai